# 05. NV5: Dự báo nhu cầu sạc và phụ tải trạm sạc (Demand & Load Forecasting)

## Mục tiêu nghiên cứu (NV5)
1. Tổng hợp chuỗi thời gian phụ tải sạc xe điện theo giờ (Hourly Load & Sessions Demand).
2. Phân tích tính mùa vụ (Seasonality) theo giờ trong ngày, ngày trong tuần và các tháng trong năm.
3. Xây dựng mô hình dự báo nhu cầu phụ tải:
   - Baseline models: Lag features, Rolling window.
   - Machine Learning models: Random Forest Regressor, XGBoost/LightGBM.
4. Đánh giá độ chính xác mô hình theo MAE, RMSE và MAPE.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.config import SESSIONS_FEATURES_PARQUET
from src.evaluation import evaluate_forecast_metrics
from src.visualization import set_plotting_style

set_plotting_style()
df = pd.read_parquet(SESSIONS_FEATURES_PARQUET)
valid = df[df['is_valid_session']].copy()
print(f"Số lượng phiên hợp lệ cho dự báo chuỗi thời gian: {len(valid):,}")


### 1. Tổng hợp dữ liệu chuỗi thời gian theo giờ (Hourly Aggregation)


In [ ]:
valid['start_hour_dt'] = valid['start_datetime_utc'].dt.floor('h')

hourly_demand = valid.groupby('start_hour_dt').agg(
    total_energy_kwh=('Energy (kWh)', 'sum'),
    session_count=('Energy (kWh)', 'count')
).reset_index()

hourly_demand = hourly_demand.sort_values(by='start_hour_dt').reset_index(drop=True)
print(f"Số mốc thời gian theo giờ: {len(hourly_demand):,} giờ")
display(hourly_demand.head())


### 2. Trực quan hóa hình mẫu nhu cầu sạc theo giờ trong ngày và ngày trong tuần


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Theo giờ trong ngày
hourly_avg = valid.groupby('start_hour')['Energy (kWh)'].mean()
sns.barplot(x=hourly_avg.index, y=hourly_avg.values, ax=axes[0], color='#2ca02c')
axes[0].set_title('Năng lượng sạc trung bình theo giờ trong ngày (UTC)')
axes[0].set_xlabel('Giờ trong ngày')
axes[0].set_ylabel('Điện năng (kWh)')

# Theo ngày trong tuần
dow_avg = valid.groupby('start_day_name')['Energy (kWh)'].mean()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.barplot(x=dow_avg.reindex(days_order).index, y=dow_avg.reindex(days_order).values, ax=axes[1], color='#ff7f0e')
axes[1].set_title('Năng lượng sạc trung bình theo ngày trong tuần')
axes[1].set_xlabel('Thứ trong tuần')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


### 3. Thiết lập đặc trưng trễ (Lag Features) và Huấn luyện mô hình Random Forest


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Tạo lag features cho chuỗi theo giờ
hourly_ts = hourly_demand.copy()
for lag in [1, 2, 24, 168]:  # 1 giờ trước, 2 giờ trước, 1 ngày trước (24h), 1 tuần trước (168h)
    hourly_ts[f'lag_{lag}'] = hourly_ts['total_energy_kwh'].shift(lag)

hourly_ts = hourly_ts.dropna().reset_index(drop=True)

feature_cols = [c for c in hourly_ts.columns if c.startswith('lag_')]
target_col = 'total_energy_kwh'

train_size = int(len(hourly_ts) * 0.8)
train = hourly_ts.iloc[:train_size]
test = hourly_ts.iloc[train_size:]

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(train[feature_cols], train[target_col])

preds = rf.predict(test[feature_cols])
metrics = evaluate_forecast_metrics(test[target_col].values, preds)

print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH DỰ BÁO PHỤ TẢI (RANDOM FOREST):")
for m_name, m_val in metrics.items():
    print(f" - {m_name}: {m_val:.3f}")
